In [10]:
from pathlib import Path
from collections import Counter

import pandas as pd
from PIL import Image
from tqdm import tqdm

In [11]:
PROJECT_ROOT = Path("/Users/user/Deep-Learning-PCB-Defect-Detection")

DATA_ROOT = PROJECT_ROOT / "data"

DEEPPCB_ROOT = DATA_ROOT / "DeepPCB"
PKU_ROOT = DATA_ROOT / "PKU-Market-PCB (Data enhanced version)"

STAGE2_DIR = PROJECT_ROOT / "outputs" / "stage2"

CROP_DIR = STAGE2_DIR / "processed_images"

METADATA_DIR = STAGE2_DIR / "metadata"

CROP_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)


print("Project root :", PROJECT_ROOT)
print("DeepPCB      :", DEEPPCB_ROOT)
print("PKU          :", PKU_ROOT)
print("Stage 2      :", STAGE2_DIR)

Project root : /Users/user/Deep-Learning-PCB-Defect-Detection
DeepPCB      : /Users/user/Deep-Learning-PCB-Defect-Detection/data/DeepPCB
PKU          : /Users/user/Deep-Learning-PCB-Defect-Detection/data/PKU-Market-PCB (Data enhanced version)
Stage 2      : /Users/user/Deep-Learning-PCB-Defect-Detection/outputs/stage2


In [12]:
# CONFIGURATION
IMAGE_SIZE = (224, 224)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
}


In [ ]:
# 3. UNIFIED CLASS MAPPING
# DeepPCB
UNIFIED_CLASSES = {
    0: "Missing_hole",
    1: "Mouse_bite",
    2: "Open_circuit",
    3: "Short",
    4: "Spur",
    5: "Spurious_copper"
}

In [14]:
DEEPPCB_TO_UNIFIED = {
    1: 2,  # open      -> Open_circuit
    2: 3,  # short     -> Short
    3: 1,  # mousebite -> Mouse_bite
    4: 4,  # spur      -> Spur
    5: 5,  # copper    -> Spurious_copper
    6: 0   # pin-hole  -> Missing_hole
}

In [15]:
# PKU
PKU_TO_UNIFIED = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5
}


PKU_CLASS_NAME_TO_ID = {
    "missing_hole": 0,
    "mouse_bite": 1,
    "open_circuit": 2,
    "short": 3,
    "spur": 4,
    "spurious_copper": 5
}

In [16]:
# HELPER FUNCTIONS

def get_image_files(image_dir):
    """
    Return all image files from a directory.
    """

    image_dir = Path(image_dir)

    if not image_dir.exists():
        return []

    return sorted([
        path
        for path in image_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ])


def get_pku_class_from_filename(filename):
    """
    Extract PKU defect class from the image filename.

    Example:
        01_missing_hole_01_1.jpg -> 0
        01_mouse_bite_01_1.jpg   -> 1
        01_open_circuit_04_1.jpg -> 2
    """

    filename = Path(filename).stem.lower()

    for class_name, class_id in PKU_CLASS_NAME_TO_ID.items():

        if class_name in filename:
            return class_id

    return None


def parse_deeppcb_label(label_path):
    """
    Read a DeepPCB annotation file.

    Format:
        x1 y1 x2 y2 class_id
    """

    annotations = []

    label_path = Path(label_path)

    if not label_path.exists():
        return annotations

    with open(label_path, "r") as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            values = line.split()

            if len(values) < 5:
                continue

            x1, y1, x2, y2, class_id = values[:5]

            annotations.append({
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
                "class_id": int(class_id)
            })

    return annotations


def validate_bbox(x1, y1, x2, y2, image_width, image_height):
    """
    Clip bounding box to image boundaries
    and check whether it is valid.
    """

    x1 = max(0, min(int(round(x1)), image_width))
    y1 = max(0, min(int(round(y1)), image_height))

    x2 = max(0, min(int(round(x2)), image_width))
    y2 = max(0, min(int(round(y2)), image_height))

    valid = (
        x2 > x1
        and
        y2 > y1
    )

    return x1, y1, x2, y2, valid

In [17]:
# PREPROCESS DEEPPCB


def preprocess_deeppcb():

    print("\n" + "=" * 70)
    print("PROCESSING DEEPPCB")
    print("=" * 70)

    metadata = []

    total_images = 0
    total_labels = 0
    total_crops = 0

    for split in ["train", "valid", "test"]:

        image_dir = DEEPPCB_ROOT / split / "images"
        label_dir = DEEPPCB_ROOT / split / "labels"

        print(f"\nDeepPCB - {split}")

        image_files = get_image_files(image_dir)

        print("Images:", len(image_files))

        for image_path in tqdm(
            image_files,
            desc=f"DeepPCB {split}"
        ):

            total_images += 1

            label_path = label_dir / f"{image_path.stem}.txt"

            # ------------------------------------------------
            # Skip image if matching annotation does not exist
            # ------------------------------------------------

            if not label_path.exists():
                continue

            total_labels += 1

            annotations = parse_deeppcb_label(label_path)

            if len(annotations) == 0:
                continue

            # ------------------------------------------------
            # Open image
            # ------------------------------------------------

            try:

                image = Image.open(image_path).convert("RGB")

            except Exception as error:

                print(
                    f"\nCannot open image: {image_path}"
                )

                print(error)

                continue

            image_width, image_height = image.size

            # ------------------------------------------------
            # Process each defect
            # ------------------------------------------------

            for defect_id, annotation in enumerate(
                annotations,
                start=1
            ):

                original_class_id = annotation["class_id"]

                unified_class_id = DEEPPCB_TO_UNIFIED.get(
                    original_class_id
                )

                if unified_class_id is None:
                    continue

                class_name = UNIFIED_CLASSES[
                    unified_class_id
                ]

                # --------------------------------------------
                # Validate bounding box
                # --------------------------------------------

                x1, y1, x2, y2, valid = validate_bbox(
                    annotation["x1"],
                    annotation["y1"],
                    annotation["x2"],
                    annotation["y2"],
                    image_width,
                    image_height
                )

                if not valid:
                    continue

                # --------------------------------------------
                # Crop defect
                # --------------------------------------------

                crop = image.crop(
                    (x1, y1, x2, y2)
                )

                # --------------------------------------------
                # Resize
                # --------------------------------------------

                crop = crop.resize(
                    IMAGE_SIZE,
                    Image.Resampling.LANCZOS
                )

                # --------------------------------------------
                # Create output directory
                # --------------------------------------------

                output_dir = (
                    CROP_DIR
                    / split
                    / class_name
                )

                output_dir.mkdir(
                    parents=True,
                    exist_ok=True
                )

                # --------------------------------------------
                # Output filename
                # --------------------------------------------

                output_name = (
                    f"DeepPCB_"
                    f"{split}_"
                    f"{image_path.stem}_"
                    f"defect_{defect_id}.png"
                )

                output_path = (
                    output_dir
                    / output_name
                )

                crop.save(output_path)

                total_crops += 1

                # --------------------------------------------
                # Save metadata
                # --------------------------------------------

                metadata.append({

                    "dataset": "DeepPCB",

                    "split": split,

                    "original_image":
                        image_path.name,

                    "label_file":
                        label_path.name,

                    "defect_id":
                        defect_id,

                    "original_class_id":
                        original_class_id,

                    "unified_class_id":
                        unified_class_id,

                    "class_name":
                        class_name,

                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,

                    "original_width":
                        image_width,

                    "original_height":
                        image_height,

                    "processed_width":
                        IMAGE_SIZE[0],

                    "processed_height":
                        IMAGE_SIZE[1],

                    "source_type":
                        "bbox_crop",

                    "image_path":
                        str(output_path)
                })

    print("\nDeepPCB summary")
    print("-----------------------------")
    print("Images       :", total_images)
    print("Matched label:", total_labels)
    print("Defect crops :", total_crops)

    return pd.DataFrame(metadata)

In [18]:
# PREPROCESS PKU


def preprocess_pku():

    print("\n" + "=" * 70)
    print("PROCESSING PKU")
    print("=" * 70)

    metadata = []

    total_images = 0
    processed_images = 0
    unknown_images = 0

    for split in ["train", "valid", "test"]:

        image_dir = PKU_ROOT / split / "images"

        print(f"\nPKU - {split}")

        image_files = get_image_files(image_dir)

        print("Images:", len(image_files))

        for image_path in tqdm(
            image_files,
            desc=f"PKU {split}"
        ):

            total_images += 1

            # ------------------------------------------------
            # Get class from filename
            # ------------------------------------------------

            original_class_id = (
                get_pku_class_from_filename(
                    image_path.name
                )
            )

            if original_class_id is None:

                unknown_images += 1

                continue

            unified_class_id = PKU_TO_UNIFIED.get(
                original_class_id
            )

            if unified_class_id is None:
                continue

            class_name = UNIFIED_CLASSES[
                unified_class_id
            ]

            # ------------------------------------------------
            # Open image
            # ------------------------------------------------

            try:

                image = Image.open(
                    image_path
                ).convert("RGB")

            except Exception as error:

                print(
                    f"\nCannot open image: {image_path}"
                )

                print(error)

                continue

            original_width, original_height = (
                image.size
            )

            # ------------------------------------------------
            # Resize
            # ------------------------------------------------

            image = image.resize(
                IMAGE_SIZE,
                Image.Resampling.LANCZOS
            )

            # ------------------------------------------------
            # Create output directory
            # ------------------------------------------------

            output_dir = (
                CROP_DIR
                / split
                / class_name
            )

            output_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            # ------------------------------------------------
            # Output filename
            # ------------------------------------------------

            output_name = (
                f"PKU_"
                f"{split}_"
                f"{image_path.stem}.png"
            )

            output_path = (
                output_dir
                / output_name
            )

            image.save(output_path)

            processed_images += 1

            # ------------------------------------------------
            # Save metadata
            # ------------------------------------------------

            metadata.append({

                "dataset": "PKU",

                "split": split,

                "original_image":
                    image_path.name,

                "label_file":
                    None,

                "defect_id":
                    None,

                "original_class_id":
                    original_class_id,

                "unified_class_id":
                    unified_class_id,

                "class_name":
                    class_name,

                "x1": None,
                "y1": None,
                "x2": None,
                "y2": None,

                "original_width":
                    original_width,

                "original_height":
                    original_height,

                "processed_width":
                    IMAGE_SIZE[0],

                "processed_height":
                    IMAGE_SIZE[1],

                "source_type":
                    "original_image",

                "image_path":
                    str(output_path)
            })

    print("\nPKU summary")
    print("-----------------------------")
    print("Images          :", total_images)
    print("Processed       :", processed_images)
    print("Unknown class   :", unknown_images)

    return pd.DataFrame(metadata)

In [19]:
# RUN PREPROCESSING

print("\nStarting Stage 2 preprocessing...")

deeppcb_metadata = preprocess_deeppcb()

pku_metadata = preprocess_pku()


Starting Stage 2 preprocessing...

PROCESSING DEEPPCB

DeepPCB - train
Images: 107


DeepPCB train: 100%|██████████| 107/107 [00:01<00:00, 57.51it/s]



DeepPCB - valid
Images: 150


DeepPCB valid: 100%|██████████| 150/150 [00:04<00:00, 32.06it/s]



DeepPCB - test
Images: 150


...


PKU - valid
Images: 0


PKU valid: 0it [00:00, ?it/s]



PKU - test
Images: 0


PKU test: 0it [00:00, ?it/s]


PKU summary
-----------------------------
Images          : 0
Processed       : 0
Unknown class   : 0


In [20]:
# COMBINE METADATA


preprocessing_metadata = pd.concat(
    [
        deeppcb_metadata,
        pku_metadata
    ],
    ignore_index=True
)

In [22]:
# SAVE METADATA

metadata_path = (
    METADATA_DIR
    / "preprocessing_metadata.csv"
)

preprocessing_metadata.to_csv(
    metadata_path,
    index=False
)


print("\n" + "=" * 70)
print("STAGE 2 COMPLETED")
print("=" * 70)

print(
    "Total processed images:",
    len(preprocessing_metadata)
)

print(
    "Metadata saved to:",
    metadata_path
)


STAGE 2 COMPLETED
Total processed images: 2374
Metadata saved to: /Users/user/Deep-Learning-PCB-Defect-Detection/outputs/stage2/metadata/preprocessing_metadata.csv


In [23]:
# DATA DISTRIBUTION

print("\nDistribution by dataset and split:")

distribution = (
    preprocessing_metadata
    .groupby(
        [
            "dataset",
            "split"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

print(distribution)


print("\nDistribution by class:")

class_distribution = (
    preprocessing_metadata
    .groupby(
        [
            "split",
            "class_name"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

print(class_distribution)


Distribution by dataset and split:
   dataset  split  count
0  DeepPCB   test    984
1  DeepPCB  train    385
2  DeepPCB  valid   1005

Distribution by class:
    split       class_name  count
0    test     Missing_hole    136
1    test       Mouse_bite    194
2    test     Open_circuit    191
3    test            Short    170
4    test             Spur    153
5    test  Spurious_copper    140
6   train     Missing_hole     67
7   train       Mouse_bite     85
8   train     Open_circuit     60
9   train            Short     44
10  train             Spur     64
11  train  Spurious_copper     65
12  valid     Missing_hole    147
13  valid       Mouse_bite    189
14  valid     Open_circuit    189
15  valid            Short    167
16  valid             Spur    165
17  valid  Spurious_copper    148


In [24]:
# FINAL CLASS DISTRIBUTION


print("\nFinal class distribution:")

final_distribution = (
    preprocessing_metadata
    .groupby("class_name")
    .size()
    .sort_index()
)

print(final_distribution)


Final class distribution:
class_name
Missing_hole       350
Mouse_bite         468
Open_circuit       440
Short              381
Spur               382
Spurious_copper    353
dtype: int64


In [25]:
# CHECK PROCESSED IMAGE FORMAT

print("\nChecking processed images...")

sample_size = min(
    20,
    len(preprocessing_metadata)
)

if sample_size > 0:

    sample_metadata = (
        preprocessing_metadata
        .sample(
            sample_size,
            random_state=42
        )
    )

    for image_path in sample_metadata[
        "image_path"
    ]:

        image_path = Path(image_path)

        image = Image.open(
            image_path
        )

        print(
            f"{image_path.name:60s}"
            f" -> size={image.size},"
            f" mode={image.mode}"
        )



Checking processed images...
DeepPCB_test_01_PCB__683_defect_6.png                        -> size=(224, 224), mode=RGB
DeepPCB_test_01_PCB__31_defect_5.png                         -> size=(224, 224), mode=RGB
DeepPCB_train_01_PCB__506_defect_12.png                      -> size=(224, 224), mode=RGB
DeepPCB_test_01_PCB__828_defect_4.png                        -> size=(224, 224), mode=RGB
DeepPCB_train_01_PCB__482_defect_2.png                       -> size=(224, 224), mode=RGB
DeepPCB_test_01_PCB__155_defect_7.png                        -> size=(224, 224), mode=RGB
DeepPCB_test_01_PCB__908_defect_5.png                        -> size=(224, 224), mode=RGB
DeepPCB_valid_01_PCB__1301_defect_3.png                      -> size=(224, 224), mode=RGB
DeepPCB_valid_01_PCB__626_defect_4.png                       -> size=(224, 224), mode=RGB
DeepPCB_test_01_PCB__588_defect_8.png                        -> size=(224, 224), mode=RGB
DeepPCB_valid_01_PCB__1104_defect_7.png                      -> size=(

In [26]:
# CHECK OUTPUT DIRECTORIES


print("\nOutput structure:")

for split in ["train", "valid", "test"]:

    split_dir = CROP_DIR / split

    if not split_dir.exists():
        continue

    print(f"\n{split}/")

    for class_dir in sorted(split_dir.iterdir()):

        if class_dir.is_dir():

            count = len(
                get_image_files(class_dir)
            )

            print(
                f"    {class_dir.name:<20}"
                f": {count}"
            )


Output structure:

train/
    Missing_hole        : 67
    Mouse_bite          : 85
    Open_circuit        : 60
    Short               : 44
    Spur                : 64
    Spurious_copper     : 65

valid/
    Missing_hole        : 147
    Mouse_bite          : 189
    Open_circuit        : 189
    Short               : 167
    Spur                : 165
    Spurious_copper     : 148

test/
    Missing_hole        : 136
    Mouse_bite          : 194
    Open_circuit        : 191
    Short               : 170
    Spur                : 153
    Spurious_copper     : 140
